# Load data from training and create pipeline for cleaning data
I will simulate loading 80% of the data at once. And the make sure the data is ok. I will then create a new schema to store the clean data

## Loading 80% of the data

In [0]:
df = spark.read.table("predictivemaintenancecatalog.raw_data.ai_4_i_2020")
display(df)

## Split data

### Stratify split

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_pd = df.toPandas()

historical_stratify_df, streaming_stratify_df = train_test_split(
    df_pd,
    test_size=0.20,
    random_state=42,
    stratify=df_pd["Machine failure"]
)

# Replay records in a reproducible sequence
streaming_stratify_df = streaming_stratify_df.sort_values("UDI")

print(historical_stratify_df.shape)
print(streaming_stratify_df.shape)

In [0]:
import pandas as pd
df_pd.columns 

In [0]:
import pandas as pd
df_pd.columns 

numerical_cols= ['Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF']
catergorical_cols = ['Product ID', 'Type',]

#### BOX PLOT

In [0]:
%pip install seaborn

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

for col in numerical_cols:
    data_dict = {
        "Train Set": historical_stratify_df[col],
        "Test Set": streaming_stratify_df[col],
    }

    plt.figure(figsize=(6, 5))
    sns.boxplot(data=data_dict)

    plt.title(f"Quick Feature Comparison: {col}")
    plt.ylabel(col)
    plt.show()

### Sequence split

In [0]:
import pandas as pd


df_pd = df.toPandas()

historical_seq_df=df_pd[:8000]
streaming_seq_df=df_pd[8000:]

print(historical_seq_df.shape)
print(streaming_seq_df.shape)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

for col in numerical_cols:
    data_dict = {
        "Train Set": historical_seq_df[col],
        "Test Set": streaming_seq_df[col],
    }

    plt.figure(figsize=(6, 5))
    sns.boxplot(data=data_dict)

    plt.title(f"Feature Comparison Sequence Split: {col}")
    plt.ylabel(col)
    plt.show()

### Use stratified split
Use 

In [0]:
# change the historical dataframe back to sparktables
historical_stratify_df_spark = spark.createDataFrame(historical_stratify_df)
streaming_stratify_df_spark = spark.createDataFrame(streaming_stratify_df)

### add meta data
Use this best practice <br>
 _ingest_timestamp: when did this batch arrive <br>
 _source: which feed produced this record <br>
 _source_file: which file specifically (important for debugging) <br>
 Underscore prefix is convention: pipeline-generated columns, not source data <br>

In [0]:
from pyspark.sql import functions as F

historical_stratify_df_spark = (
    historical_stratify_df_spark
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source", F.lit("historical_batch"))
    .withColumn("_source_file", F.lit("ai4i2020.csv"))
)
display(historical_stratify_df_spark)

## Write the historical data to the bronze

In [0]:
spark.sql("Create schema if not exists predictivemaintenancecatalog.bronze_data_not_quality_checked")

clean_cols = [c.replace(" ", "_").replace("[", "").replace("]", "") for c in historical_stratify_df_spark.columns]
historical_stratify_df_spark.toDF(*clean_cols).write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("predictivemaintenancecatalog.bronze_data_not_quality_checked.historical_stratified")


In [0]:
# overwrite the original table with the enriched historical data (keeps _ingest_timestamp, _source, _source_file)

# Clean column names — Delta does not allow spaces or [ ] in column names
clean_cols = [c.replace(" ", "_").replace("[", "").replace("]", "") for c in historical_stratify_df_spark.columns]
df_clean = historical_stratify_df_spark.toDF(*clean_cols)

# 1. Create a temporary view of your current dataframe
df_clean.createOrReplaceTempView("temp_df")

# 2. Run SQL to completely replace the table and its schema
spark.sql("""
    CREATE OR REPLACE TABLE predictivemaintenancecatalog.bronze_data_not_quality_checked.historical_stratified
    AS SELECT * FROM temp_df
""")


df_clean.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.bronze_data_not_quality_checked.historical_stratified")